# الدرس 18: تأمين وكلاء الذكاء الاصطناعي بإيصالات تشفيرية

## دفتر التمارين العملي

يمر هذا الدفتر بأربعة تمارين:

1. **توقيع إيصالك الأول** لمكالمة أداة الوكيل والتحقق منه.
2. **التلاعب بالإيصال** ومراقبة فشل التحقق.
3. **بناء سلسلة مكونة من ثلاثة إيصالات** وتأكيد سلامة السلسلة.
4. **تغليف مكالمة أداة إطار عمل Microsoft Agent** بحيث ينبعث إيصال عن كل إجراء.

جميع الأدوات التشفيرية مستوردة من مكتبات مُعتنى بها جيدًا (`pynacl` لـ Ed25519، و`jcs` للـ JSON التوافقي RFC 8785، و`hashlib` من مكتبة بايثون القياسية لـ SHA-256). منطق الإيصال نفسه هو بايثون بسيط يمكنك قراءته وتعديله.

شغّل الخلايا بالترتيب. كل قسم قصير ومكتفي ذاتيًا.


## الإعداد

قم بتثبيت الاعتماديتين. كلاهما يحمل تراخيص متساهلة (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## أدوات مساعدة

هاتان الأداتان المساعدتان تتعاملان مع ترميز base64url (بدون تعبئة) وتجزئة SHA-256 للكائنات العشوائية. تبقيان بقية الدفتر مركزة على منطق الإيصال نفسه.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## القسم 1: وقع إيصالك الأول

تخيل أن وكيلنا لـ **Contoso Travel** قد بحث للتو عن رحلات من سيدني إلى لوس أنجلوس لعميل. نريد تسجيل هذه المكالمة كإيصال موقع حتى يتمكن مدقق حسابات مستقبلي من التحقق منه دون الاعتماد علينا.

### الخطوة 1.1: إنشاء مفتاح توقيع

في البيئة الإنتاجية، يكون مفتاح التوقيع الخاص بالوكيل مخزنًا في وحدة أمان الأجهزة (HSM)، أو Azure Key Vault، أو مخزن محمي مماثل. لهذا الدرس، ننشئ مفتاحًا جديدًا في الذاكرة.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### الخطوة 1.2: بناء حمولة الإيصال

تحتوي الحمولة على كل شيء نريد أن يشهد عليه الإيصال: من الذي تصرف، وأي أداة، وبأي وسائط، وما الذي تم إرجاعه، وتحت أي سياسة، ومتى. نقوم بعمل تجزئة للوسائط والنتيجة بدلاً من تضمينها مباشرة لمنع تسرب محتوى حساس عبر الإيصال.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### الخطوة 1.3: توقيع وتجميع الإيصال

ثلاث خطوات:

1. تحويل الحمولة إلى الشكل القانوني باستخدام JCS بحيث ينتج تنفيذان مختلفان نفس الإيصال المنطقي بايتات متطابقة تمامًا.
2. تجزئة البايتات القانونية باستخدام SHA-256.
3. توقيع التجزئة باستخدام المفتاح الخاص Ed25519.

ثم يتم إرفاق التوقيع بالحمولة الأصلية لإنتاج الإيصال النهائي.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### الخطوة 1.4: التحقق من الإيصال

التحقق يعكس العملية. نقوم بإزالة التوقيع، وإعادة حساب الهاش القانوني، والتحقق من التوقيع مقابل المفتاح العام في الإيصال.

المدقق الذي يقوم بهذا التحقق لا يحتاج منا سوى الإيصال نفسه. لا تحتاج إلى استدعاء خدمة، ولا الاستعلام عن دليل المفاتيح، ولا الثقة مطلوبة.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

يجب أن ترى `الإيصال صالح: صحيح`. لقد قام الوكيل بإنتاج أول سجل تدقيق موقع تشفيرياً.  


## القسم 2: العبث بالإيصال

الغرض الكامل من الإيصالات هو أنها تكشف عن العبث بها. دعونا نثبت ذلك.

سنقوم بتعديل حرف واحد فقط من الإيصال ونراقب فشل التحقق.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### ماذا حدث للتو؟

عندما قمنا بتغيير `policy_id`، تغيرت البايتات الأصلية. تغيرت قيمة الـ SHA-256 لتلك البايتات. التوقيع (الذي كان على قيمة الهاش الأصلية) لم يعد يطابق الهاش الجديد. التحقق يعيد القيمة `False` بشكل صحيح.

لا يمكن تعديل أي حقل من الحقول في الإيصال مع إمكانية تحقق التوقيع، إلا إذا كان المهاجم يمتلك المفتاح الخاص. طالما أن المفتاح الخاص محفوظ في خزنة مفاتيح والمفتاح العام منشور، فإن التلاعب مستحيل أن يُخفي.

جرب بنفسك: عدّل `tool_name` أو `agent_id` أو `timestamp` في الخلية أعلاه وأعد التشغيل. كل تعديل ينتج إيصال غير صالح.


## القسم 3: ربط الإيصالات معًا

يحمي إيصال واحد إجراءً واحدًا. يتخذ معظم الوكلاء العديد من الإجراءات. لجعل تسلسل الأحداث بأكمله واضح التلاعب، نربط كل إيصال بالإيصال السابق من خلال تضمين تجزئة الإيصال السابق في حمولة الإيصال الجديد.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

إذا قام أي شخص بإزالة إيصال أو إعادة ترتيبه، ينكسر السلسلة عند تلك النقطة بالضبط. تفشل عملية التحقق من أي إيصال لاحق لأن `previous_receipt_hash` لم يعد يطابق التجزئة الفعلية لسابقه.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

الآن اكسر السلسلة عن طريق التلاعب بالإيصال الأوسط وأعد التحقق. يفشل الإيصال المتلاعب في فحص التوقيع، ويفشل الإيصال التالي في فحص رابط السلسلة (لأن قيمة `previous_receipt_hash` لم تعد تتطابق مع هاش الإيصال الأوسط المعدل).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

الإيصال 0 لا يزال يتحقق (لم يتم تعديله وليس له سابق يعتمد عليه). يفشل الإيصال 1 في فحص التوقيع الخاص به لأننا قمنا بتغيير `tool_args_hash`. يفشل الإيصال 2 في فحص رابط السلسلة لأن `previous_receipt_hash` الخاص به تم حسابه بناءً على الإيصال الأصلي (المعدل الآن) 1.

حتى إذا قام المهاجم بإعادة توقيع الإيصال 1 المعدل (وهو ما لا يمكنه فعله بدون المفتاح الخاص)، فإن عدم تطابق رابط السلسلة في الإيصال 2 سيكشف التلاعب. لإخفاء التغيير، سيضطر المهاجم إلى إعادة توقيع كل إيصال من نقطة التعديل فصاعدًا، وهذا يتطلب امتلاك المفتاح الخاص.


## القسم 4: تغليف استدعاء أداة العميل بتوقيع الإيصال

في النشر الحقيقي، لا تريد أن يتذكر كل مؤلف عميل استدعاء `make_receipt`. تريد أن يكون توقيع الإيصال تلقائيًا في كل مرة يتم استدعاء الأداة.

هنا أبسط نموذج: فئة تغليف تأخذ أي دالة أداة قابلة للاستدعاء وتعيد نسخة تصدر إيصالًا. هذا يتكيف مع أي إطار عميل، بما في ذلك إطار عمل Microsoft Agent (`agent_framework.foundry`).

إذا لم يكن لديك مشروع Microsoft Foundry مُعد، فإن المحاكاة المحلية أدناه لا تزال توضح هذا النموذج.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### التكامل مع إطار عمل وكيل مايكروسوفت

المغلف `ReceiptedTool` أعلاه مستقل عن الإطار. لاستخدامه داخل وكيل مبني بإطار عمل وكيل مايكروسوفت، قم بتسجيل الدالة المغلفة كأداة. مخطط (ستستبدل النموذج بأداة تسجيل حقيقية من Microsoft Foundry):

```python
# رمز مزيف يعرض شكل التكامل.
# استيراد os
# من agent_framework.foundry استيراد FoundryChatClient
# من azure.identity استيراد AzureCliCredential
#
# الموفر = FoundryChatClient(
#     نقطة نهاية المشروع=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     النموذج=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     الاعتماد=AzureCliCredential(),
# )
# العميل = الموفر كعميل_وكيل(
#     التعليمات="أنت وكيل سفر في Contoso ..."
#     الأدوات=[receipted_lookup],   # الأداة المغلفة، ليست الدالة الخام
# )
# الاستجابة = العميل.تشغيل("البحث عن رحلات من سيدني إلى لوس أنجلوس في يونيو.")
#
# # بعد التشغيل، كل استدعاء أداة قام به الوكيل لديه إيصال موقّع:
# سلسلة التدقيق = receipted_lookup.receipts
```

إطار عمل الوكيل لا يحتاج إلى معرفة أي شيء عن الإيصالات. توقيع الإيصال مغلف حول الأداة، وليس مدمجًا داخل الإطار. هذه هي الطريقة التي تضيف بها الأصل إلى رمز الوكيل الحالي دون إعادة كتابة الوكيل.


## ملخص وتحدي التمدد

لديك:

- تم إنشاء زوج مفاتيح Ed25519.
- تم بناء وتوقيع إيصال لاستدعاء أداة الوكيل.
- تم التحقق من الإيصال دون اتصال باستخدام المفتاح العام فقط.
- تم العبث بالإيصال ولوحظ فشل التحقق.
- تم بناء تسلسل مرتبط بواسطة تجزئة لثلاث إيصالات.
- تم العبث بمنتصف السلسلة ولوحظ فشل في كل من التوقيع وفشل رابط السلسلة.
- تم تغليف دالة الأداة مع توقيع الإيصال تلقائيًا.

**تحدي التمدد.** وسّع مخطط الإيصال بحقل `request_id` (معرّف UUID لتعقب موزع). حدِّث `make_receipt` ليشمله، وتأكد من أن الإيصالات لا تزال تتحقق نهاية إلى نهاية. ثم عدل الحقل بعد التوقيع وتأكد أن التحقق يفشل. هذا يجبرك على استيعاب كيف يساهم كل بايت من الترميز القانوني في التوقيع.

**حدود مهمة.** تثبت الإيصالات ثلاث أشياء فقط: النسبة (هذا المفتاح وقع هذا المحتوى)، النزاهة (المحتوى لم يتغير منذ التوقيع)، والترتيب (هذا الإيصال جاء بعد ذلك الإيصال). إنها لا تثبت أن إجراء الوكيل كان صحيحًا، أو أن السياسة المسماة في `policy_id` قد تم تقييمها بالفعل، أو أن الوكيل اتبع كل قاعدة. الإيصالات هي الأساس. الحوكمة هي النظام الذي تبنيه فوقها.

اقرأ ملف README للدرس مرة أخرى مع وضع هذه الحدود في الاعتبار. الخطأ الأكثر شيوعًا الذي ترتكبه الفرق مع الإيصالات هو افتراض أن "لدينا إيصالات" تعني "نحن محكمون". هذا غير صحيح. تجعل الإيصالات سلوك الوكيل قابلاً للتدقيق. لكنها لا تجعله صحيحًا.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
